In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

options = Options()
options.headless = False
driver = webdriver.Firefox(service=Service(), options=options)

project_data = []
max_pages = 104

for page in range(1, max_pages + 1):
    url = f"https://www.kwf.nl/onderzoek/onderzoeksdatabase?page={page}"
    print(f"📄 Pagina {page} ophalen...")
    driver.get(url)
    time.sleep(2)

    containers = driver.find_elements(By.CSS_SELECTOR, "a.doc-result-project-title")

    for el in containers:
        try:
            parent = el.find_element(By.XPATH, "./parent::div")

            title = el.text.strip()
            link = el.get_attribute("href")
            if link.startswith("/"):
                link = "https://www.kwf.nl" + link

            onderzoeker = parent.find_element(By.CLASS_NAME, "doc-result-project-leader").text.strip()
            organisatie = parent.find_element(By.CLASS_NAME, "doc-result-project-institute").text.strip()
            samenvatting = parent.find_element(By.CLASS_NAME, "doc-result-project-summary").text.strip()
            projectnummer = parent.find_element(By.CLASS_NAME, "doc-result-project-number").text.replace("Projectnummer: ", "").strip()
            startjaar = parent.find_element(By.CLASS_NAME, "doc-result-start-year").text.replace("Startjaar: ", "").strip()
            thema = parent.find_element(By.CLASS_NAME, "doc-result-project-theme").text.strip()

            try:
                status = parent.find_element(By.CLASS_NAME, "doc-result-project-status").text.strip()
            except:
                status = ""

            try:
                gericht_geven = "Ja" if parent.find_element(By.CLASS_NAME, "doc-result-suitable-for-targeted-gift") else "Nee"
            except:
                gericht_geven = "Nee"

            project_data.append({
                "title": title,
                "url": link,
                "onderzoeker": onderzoeker,
                "organisatie": organisatie,
                "samenvatting": samenvatting,
                "projectnummer": projectnummer,
                "startjaar": startjaar,
                "thema": thema,
                "status": status,
                "gericht_geven": gericht_geven
            })

        except Exception as e:
            print(f"⚠️ Fout bij project: {e}")

driver.quit()

df = pd.DataFrame(project_data)
print(f"\n✅ {len(df)} projecten verzameld.")
df.head()


In [ ]:
import pandas as pd
import re

# Je NKA-trefwoorden
nka_keywords = {
    "1. Roken": ["roken", "tabak", "nicotine", "rookvrije generatie"],
    "2. Alcoholconsumptie": ["alcohol", "alcoholgebruik", "drankgebruik"],
    "3. UV-straling": ["uv-straling", "zon", "zonnebrand", "melanoom", "huidkanker"],
    "4. Infectieziekten": ["hpv", "hepatitis", "infectie", "virus", "vaccinatie"],
    "5. Milieu en luchtverontreiniging": ["luchtverontreiniging", "milieu", "fijnstof", "vervuiling", "asbest", "pfas", "benzeen"],
    "6. Voeding en beweging": ["voeding", "eten", "fruit", "groente", "bewegen", "obesitas", "overgewicht", "leefstijl"],
    "7. Erfelijke belasting": ["erfelijk", "genetisch", "dna", "mutatie"],
    "8. Vroege opsporing": ["bevolkingsonderzoek", "screening", "vroeg", "detectie", "opsporing"],
    "9. Diagnostiek": ["diagnose", "diagnostiek", "beeldvorming", "screening"],
    "10. Behandeling": ["behandeling", "therapie", "bestraling", "chemotherapie", "immuuntherapie", "vaccin"],
    "11. Innovaties": ["innovatie", "ai", "kunstmatige intelligentie", "nanotechnologie", "nieuwe techniek", "onderzoek"],
    "12. Zeldzame kankers": ["zeldzame kanker", "tumor onbekend", "aya", "sarcoom", "zeldzaam"],
    "13. Palliatieve zorg": ["palliatieve", "levenseinde", "sterven", "comfortzorg"],
    "14. Leefstijlondersteuning": ["leefstijl", "gezondheidsgedrag", "ondersteuning", "coach", "begeleiding"],
    "15. Late gevolgen": ["late gevolgen", "bijwerkingen", "langetermijneffecten"],
    "16. Onderwijs en kanker": ["onderwijs", "school", "jongeren", "educatie", "lesmateriaal"],
    "17. Werk en kanker": ["werk", "re-integratie", "arbeid", "loopbaan", "ziekteverzuim"],
    "18. Financiële gevolgen": ["kosten", "financieel", "vergoeding", "armoede", "zorgkosten"],
    "19. Naasten en mantelzorgers": ["mantelzorger", "naasten", "partner", "gezinsleden"],
    "20. Sociaal-emotionele ondersteuning": ["psychosociaal", "angst", "emoties", "psychologie", "steun", "therapeut"]
}

# Matching-functie
def match_nka_doelen(titel, samenvatting, nka_keywords):
    tekst = f"{titel} {samenvatting}".lower()
    gevonden = []
    for doel, trefwoorden in nka_keywords.items():
        for woord in trefwoorden:
            if re.search(rf'\b{re.escape(woord)}\b', tekst):
                gevonden.append(doel)
                break  # één woord is voldoende
    return gevonden

# Toepassen op de DataFrame (df moet al bestaan met kolommen 'title' en 'samenvatting')
df["nka_doelen"] = df.apply(lambda row: match_nka_doelen(row["title"], row["samenvatting"], nka_keywords), axis=1)

# Binaire kolommen toevoegen
for doel in nka_keywords.keys():
    kolomnaam = "doel_" + doel.split(".")[0] + "_" + doel.split(".")[1].strip().lower().replace(" ", "_")
    df[kolomnaam] = df["nka_doelen"].apply(lambda doelen: 1 if doel in doelen else 0)

# Bekijken
print(f"✅ Aantal projecten met NKA-matches: {(df['nka_doelen'].apply(len) > 0).sum()} van {len(df)}")
df[["title", "nka_doelen"] + [col for col in df.columns if col.startswith("doel_")]].head(10)


In [ ]:
import circlify
import plotly.graph_objects as go
import pandas as pd
from collections import Counter

# --- KWF Data ---
alle_doelen = sum(df["nka_doelen"], [])
counts = Counter(alle_doelen)

# --- Mapping: NKA-doelnummer naar beschrijving (voor legenda) ---
nka_labels = {
    "1. Roken": 1, "2. Alcoholconsumptie": 2, "3. UV-straling": 3, "4. Infectieziekten": 4,
    "5. Milieu en luchtverontreiniging": 5, "6. Voeding en beweging": 6, "7. Erfelijke belasting": 7,
    "8. Vroege opsporing": 8, "9. Diagnostiek": 9, "10. Behandeling": 10, "11. Innovaties": 11,
    "12. Zeldzame kankers": 12, "13. Palliatieve zorg": 13, "14. Leefstijlondersteuning": 14,
    "15. Late gevolgen": 15, "16. Onderwijs en kanker": 16, "17. Werk en kanker": 17,
    "18. Financiële gevolgen": 18, "19. Naasten en mantelzorgers": 19, "20. Sociaal-emotionele ondersteuning": 20
}
nka_legenda = {v: k for k, v in nka_labels.items()}  # omgekeerd voor legenda

# --- Circlify met numerieke labels ---
data = [{"id": nka_labels[label], "datum": value} for label, value in counts.items()]
circles = circlify.circlify(data, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1))

bubble_data = []
for circle in circles:
    if circle.level == 1:
        bubble_data.append({
            'nummer': circle.ex["id"],
            'label': nka_legenda[circle.ex["id"]],
            'value': circle.ex["datum"],
            'x': circle.x,
            'y': circle.y,
            'r': circle.r
        })

df_plot = pd.DataFrame(bubble_data)

# Kleur per value
vals = df_plot['value']
norm = (vals - vals.min()) / (vals.max() - vals.min())
df_plot['color'] = [
    f"rgba({int(50 + 200 * n)}, {int(100 + 100 * n)}, 255, 0.8)" for n in norm
]

# Bereik berekenen
x_min = (df_plot["x"] - df_plot["r"]).min()
x_max = (df_plot["x"] + df_plot["r"]).max()
y_min = (df_plot["y"] - df_plot["r"]).min()
y_max = (df_plot["y"] + df_plot["r"]).max()

# --- Bubble Chart bouwen ---
fig = go.Figure()

for row in df_plot.itertuples():
    fig.add_shape(type="circle",
                  x0=row.x - row.r, y0=row.y - row.r,
                  x1=row.x + row.r, y1=row.y + row.r,
                  xref="x", yref="y",
                  line_color="white",
                  fillcolor=row.color)
    fig.add_trace(go.Scatter(
        x=[row.x],
        y=[row.y],
        text=[str(row.nummer)],
        hovertext=f"{row.label}<br>{row.value} projecten",
        mode="text",
        textposition="middle center",
        hoverinfo="text"
    ))

# --- Legenda bouwen als extra tekstspoor ---
legend_items = [f"{num}. {label.split('. ', 1)[1]}" for num, label in sorted(nka_legenda.items())]
legend_text = "<br>".join(legend_items)

# --- Layout configureren ---
fig.update_layout(
    title="KWF-projecten en de Nederlandse Kanker Agenda",
    xaxis=dict(visible=False, range=[x_min, x_max]),
    yaxis=dict(visible=False, range=[y_min, y_max], scaleanchor="x"),
    showlegend=False,
    plot_bgcolor="white",
    height=900,
    annotations=[
        dict(
            xref="paper", yref="paper",
            x=1.05, y=1,
            align="left",
            showarrow=False,
            text=f"<b>Legenda NKA-doelen:</b><br>{legend_text}",
            font=dict(size=12),
            bordercolor="black",
            borderwidth=1
        )
    ]
)

# Toon grafiek
fig.write_html("kwf_nka_bubbles_nummers_met_legenda.html")
import webbrowser
webbrowser.open("kwf_nka_bubbles_nummers_met_legenda.html")


In [ ]:
import plotly.express as px
import pandas as pd
from collections import Counter

# Aantal projecten per doel
alle_doelen = sum(df["nka_doelen"], [])
counts = Counter(alle_doelen)

# Data voorbereiden
df_polar = pd.DataFrame(counts.items(), columns=["NKA-doel", "Aantal projecten"])
df_polar["NKA-nummer"] = df_polar["NKA-doel"].str.extract(r"^(\d+)")
df_polar = df_polar.sort_values("NKA-nummer").reset_index(drop=True)

# Polar bar chart
fig = px.bar_polar(
    df_polar,
    r="Aantal projecten",
    theta="NKA-doel",
    color="Aantal projecten",
    color_continuous_scale="Viridis",
    title="Aantal KWF-projecten per NKA-doel",
    template="plotly_white"
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(showticklabels=True, ticks='', showline=False, gridcolor="gray"),
        angularaxis=dict(tickfont=dict(size=10)),
    ),
    height=700
)

fig.write_html("kwf_nka_polar_chart.html")
import webbrowser
webbrowser.open("kwf_nka_polar_chart.html")


In [ ]:
import numpy as np
import plotly.express as px

# Log10-schaal toepassen (+1 om log(0) te vermijden)
df_polar["Log aantal projecten"] = df_polar["Aantal projecten"].apply(lambda x: np.log10(x + 1))

# Plot met log-schaal als straalwaarde
fig = px.bar_polar(
    df_polar,
    r="Log aantal projecten",
    theta="NKA-doel",
    color="Aantal projecten",  # echte aantallen nog steeds voor kleur
    color_continuous_scale="Viridis",
    title="Aantal KWF-projecten per NKA-doel (log-schaal)",
    template="plotly_white"
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(showticklabels=True, ticks='', showline=False, gridcolor="lightgray"),
        angularaxis=dict(tickfont=dict(size=11)),
    ),
    height=700
)

fig.write_html("kwf_nka_polar_logscale.html")
import webbrowser
webbrowser.open("kwf_nka_polar_logscale.html")


OpenAlex 


In [ ]:
import requests
import pandas as pd

url = "https://api.openalex.org/works"
params = {
    "filter": "publication_year:2017-2025,grants.funder:f4320322777",
    "sort": "cited_by_count:desc",
    "per_page": 200,
    "cursor": "*"
}

records = []
while url:
    resp = requests.get(url, params=params)
    data = resp.json()

    for work in data["results"]:
        records.append({
            "title": work.get("display_name", ""),
            "abstract": work.get("abstract_inverted_index", None),
            "count": work.get("cited_by_count", 0),
            "id": work.get("id", ""),
            "doi": work.get("doi", ""),
            "publication_year": work.get("publication_year", ""),
            "authors": ", ".join([a["author"]["display_name"] for a in work.get("authorships", [])]),
            "journal": work.get("host_venue", {}).get("display_name", "")
        })

    meta = data.get("meta", {})
    cursor = meta.get("next_cursor")
    if not cursor:
        break
    params["cursor"] = cursor

df_api = pd.DataFrame(records)

# 👀 Verken de data
print(f"✅ Ophalen klaar, {len(df_api)} artikelen.\n")
print(df_api.head(3))
print("\n📌 Kolommen en types:")
print(df_api.dtypes)
print("\n🔍 Voorbeeld abstract:")
print(df_api["abstract"].dropna().iloc[0])


In [ ]:
# ✅ Engelse vertaling van NKA-doelen met relevante keywords
nka_keywords_en = {
    "1. Smoking": ["smoking", "tobacco", "nicotine", "smoke-free", "lung cancer"],
    "2. Alcohol consumption": ["alcohol", "drinking", "liquor"],
    "3. UV radiation": ["uv", "sunlight", "melanoma", "skin cancer"],
    "4. Infectious diseases": ["hpv", "hepatitis", "infection", "virus", "vaccination"],
    "5. Environment and pollution": ["air pollution", "environment", "fine dust", "pollution", "asbestos", "pfas", "benzene"],
    "6. Nutrition and exercise": ["nutrition", "diet", "fruit", "vegetable", "exercise", "obesity", "overweight", "lifestyle"],
    "7. Genetic predisposition": ["hereditary", "genetic", "dna", "mutation"],
    "8. Early detection": ["screening", "early", "detection"],
    "9. Diagnostics": ["diagnosis", "diagnostic", "imaging"],
    "10. Treatment": ["treatment", "therapy", "radiation", "chemotherapy", "immunotherapy", "vaccine"],
    "11. Innovation": ["innovation", "ai", "artificial intelligence", "nanotechnology", "novel technique", "research"],
    "12. Rare cancers": ["rare cancer", "unknown tumor", "aya", "sarcoma", "rare"],
    "13. Palliative care": ["palliative", "end-of-life", "dying", "comfort care"],
    "14. Lifestyle support": ["lifestyle", "health behavior", "support", "coach", "guidance"],
    "15. Late effects": ["late effects", "side effects", "long-term effects"],
    "16. Education and cancer": ["education", "school", "youth", "curriculum", "teaching material"],
    "17. Work and cancer": ["work", "reintegration", "employment", "career", "sick leave"],
    "18. Financial impact": ["cost", "financial", "reimbursement", "poverty", "healthcare costs"],
    "19. Caregivers and family": ["caregiver", "partner", "family members", "loved ones"],
    "20. Psychosocial support": ["psychosocial", "anxiety", "emotions", "psychology", "support", "therapy"]
}

# 🛠 Reconstruct abstract from inverted index
def reconstruct_abstract(index):
    if not isinstance(index, dict):
        return ""
    max_pos = max(pos for positions in index.values() for pos in positions)
    words = [""] * (max_pos + 1)
    for word, positions in index.items():
        for pos in positions:
            words[pos] = word
    return " ".join(words)

# 🎯 Match NKA goals in English
def match_nka_goals(row):
    text = row["title"].lower()
    abstract = reconstruct_abstract(row["abstract"])
    text += " " + abstract.lower()
    return [goal for goal, keywords in nka_keywords_en.items() if any(kw in text for kw in keywords)]

# 🔍 Apply matching to each row
df_api["nka_goals"] = df_api.apply(match_nka_goals, axis=1)

# 📊 Count number of articles per NKA goal
df_counts = df_api.explode("nka_goals").dropna(subset=["nka_goals"])
df_counts = df_counts.groupby("nka_goals").size().reset_index(name="Number of articles")

# ✅ Preview
print(df_counts)


In [ ]:
import circlify
import plotly.graph_objects as go
import pandas as pd
from collections import Counter

# --- NKA labels en mapping ---
nka_labels = {
    "1. Smoking": 1, "2. Alcohol consumption": 2, "3. UV radiation": 3, "4. Infectious diseases": 4,
    "5. Environment and pollution": 5, "6. Nutrition and exercise": 6, "7. Genetic predisposition": 7,
    "8. Early detection": 8, "9. Diagnostics": 9, "10. Treatment": 10, "11. Innovation": 11,
    "12. Rare cancers": 12, "13. Palliative care": 13, "14. Lifestyle support": 14,
    "15. Late effects": 15, "16. Education and cancer": 16, "17. Work and cancer": 17,
    "18. Financial impact": 18, "19. Caregivers and family": 19, "20. Psychosocial support": 20
}
nka_legenda = {v: k for k, v in nka_labels.items()}

# --- Circlify voorbereiden ---
counts = dict(zip(df_counts["nka_goals"], df_counts["Number of articles"]))
data = [{"id": nka_labels[label], "datum": value} for label, value in counts.items()]
circles = circlify.circlify(data, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1))

bubble_data = []
for circle in circles:
    if circle.level == 1:
        bubble_data.append({
            'nummer': circle.ex["id"],
            'label': nka_legenda[circle.ex["id"]],
            'value': circle.ex["datum"],
            'x': circle.x,
            'y': circle.y,
            'r': circle.r
        })

df_plot = pd.DataFrame(bubble_data)

# Kleur per value
vals = df_plot['value']
norm = (vals - vals.min()) / (vals.max() - vals.min())
df_plot['color'] = [
    f"rgba({int(50 + 200 * n)}, {int(100 + 100 * n)}, 255, 0.8)" for n in norm
]

# Bereik
x_min = (df_plot["x"] - df_plot["r"]).min()
x_max = (df_plot["x"] + df_plot["r"]).max()
y_min = (df_plot["y"] - df_plot["r"]).min()
y_max = (df_plot["y"] + df_plot["r"]).max()

# --- Plotly figuur ---
fig = go.Figure()

for row in df_plot.itertuples():
    fig.add_shape(type="circle",
                  x0=row.x - row.r, y0=row.y - row.r,
                  x1=row.x + row.r, y1=row.y + row.r,
                  xref="x", yref="y",
                  line_color="white",
                  fillcolor=row.color)
    fig.add_trace(go.Scatter(
        x=[row.x],
        y=[row.y],
        text=[str(row.nummer)],
        hovertext=f"{row.label}<br>{row.value} artikelen",
        mode="text",
        textposition="middle center",
        hoverinfo="text"
    ))

# Legenda
legend_items = [f"{num}. {label.split('. ', 1)[1]}" for num, label in sorted(nka_legenda.items())]
legend_text = "<br>".join(legend_items)

# Layout
fig.update_layout(
    title="Artikelen en de Nederlandse Kanker Agenda",
    xaxis=dict(visible=False, range=[x_min, x_max]),
    yaxis=dict(visible=False, range=[y_min, y_max], scaleanchor="x"),
    showlegend=False,
    plot_bgcolor="white",
    height=900,
    annotations=[
        dict(
            xref="paper", yref="paper",
            x=1.05, y=1,
            align="left",
            showarrow=False,
            text=f"<b>Legenda NKA-doelen:</b><br>{legend_text}",
            font=dict(size=12),
            bordercolor="black",
            borderwidth=1
        )
    ]
)

# Opslaan en openen
fig.write_html("nka_articles_bubbles.html")
import webbrowser
webbrowser.open("nka_articles_bubbles.html")




In [ ]:
import numpy as np
import plotly.express as px

# 📊 Voorbereiden polar dataframe uit df_counts
df_polar_articles = df_counts.copy()
df_polar_articles.rename(columns={"nka_goals": "NKA-doel", "Number of articles": "Aantal artikelen"}, inplace=True)

# 🧮 Log10-schaal toepassen
df_polar_articles["Log aantal artikelen"] = df_polar_articles["Aantal artikelen"].apply(lambda x: np.log10(x + 1))

# 🎯 Polar chart maken
fig = px.bar_polar(
    df_polar_articles,
    r="Log aantal artikelen",
    theta="NKA-doel",
    color="Aantal artikelen",  # originele aantallen gebruiken voor kleur
    color_continuous_scale="Viridis",
    title="Aantal artikelen per NKA-doel (log-schaal)",
    template="plotly_white"
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(showticklabels=True, ticks='', showline=False, gridcolor="lightgray"),
        angularaxis=dict(tickfont=dict(size=11)),
    ),
    height=700
)

# 💾 Opslaan en openen
fig.write_html("nka_articles_polar_logscale.html")
import webbrowser
webbrowser.open("nka_articles_polar_logscale.html")
